# Cooperation dynamics of LLM agents — source of truth

**Branch `donors-game-dynamics`, package `llm_dynamics/`.** This notebook is
regenerated by `llm_dynamics/build_notebook.py`; every table and figure below is
read from `llm_dynamics/results/` at execution time, so it always reflects the
latest runs. Narrative sections record the *decisions* (design, mappings,
deviations, caveats); code cells record the *evidence*.

## 0. What we are doing, in one paragraph

MARL-mixed studies the **dynamics of cooperation** — not just whether agents end up
cooperating, but the vector field of learning that carries them there — using
CRLD flow fields (pyCRLD) and, for learners without a closed-form flow (deep RL),
measured trajectories overlaid on the analytic flow of the reduced matrix game.
donorSim (branch `neurips-methodology`) trains LLMs with RL (verl GRPO; reward =
normalized donors-game payoff + HKB coordination term + CFE group term) on a
dyadic donors game against scripted strategies, and claims the training
**changes the LLM's dynamics**. Here we make that claim measurable: the same
phase portraits, with an LLM in the learner's seat — first the **base model**
(this notebook), then the RL-trained checkpoint on the same axes.


## 1. Background we build on

### 1.1 MARL-mixed methodology (this repo)
* State of the system = joint policy $X_i(a\mid s)$. A flow plot is a 2-D slice
  (agent-0 $P(C)$ vs agent-1 $P(C)$ in a chosen memory state); arrows are the
  deterministic CRLD learning step $\Delta X = \mathrm{step}(X)-X$
  (`pyCRLD.Utils.FlowPlot.plot_strategy_flow`, `use_RPEarrows=False`), with
  off-slice coordinates marginalized by Latin-hypercube sampling.
* Memory-$m$ games via `HistoryEmbedded(env, h=(m,)*(N+1))`, state count $(2^N)^m$.
* Observability regimes = row-stochastic observation-aliasing matrices
  (`scratch_repro/mem_obs_grids.py`: full / self / others / coop / def / blind).
* Learners with no analytic flow (JaxMARL deep RL, `jaxmarl_env/algo_phase.py`):
  grey CRLD arrows as theory, coloured measured paths as reality, **the deviation
  is the finding**.
* Paper findings we lean on: self-observability is load-bearing; memory enlarges
  the cooperative basin (memory-1 flow drains to defection, memory-2 has a (1,1)
  attractor); actor-critic is bistable, SARSA converges to an interior point.

### 1.2 donorSim `neurips-methodology`
* **Basic dyadic scenario** = the `direct_indirect` training rollout
  (`donor_sim/verl_integration/donor_game_interaction.py` +
  `donor_sim/game_helpers.py`): one LLM vs one scripted partner drawn from
  `OPPONENT_POOL` = {AllC .15, AllD .30, TFT .20, TF2T .20, Random .10, Grim .05};
  $b\sim U[3,6]$, $c/b\sim U[0.1,1.5]$, $w,q\in\{0,.25,.5,.75,1\}$, 3–8 rounds.
* Semantics: simultaneous moves ("If you COOPERATE: you receive 0, partner
  receives $b$; if you DEFECT: you receive $c$"); partner reacts to the model's
  *prior* rounds; **$w$** = probability the same partner returns next round
  (else a stranger arrives, per-partner history wiped); **$q$** = probability a
  reputation report about the partner is available.
* Training: verl GRPO (LoRA r=16, KL 0.001, 8 generations, per-turn credit),
  reward $r = \frac{\pi + c}{b+c} + \lambda_{tom}[4q\cos\phi + \tfrac{c}{b}\cos 2\phi] - \lambda_g\,\mathrm{CFE}$
  with HKB phase $\phi$ from a window-4 coordination signal; phase transition at
  $q = c/b$ (Nowak's indirect-reciprocity threshold). Released run on Qwen3-8B;
  a Qwen3-32B methodology checkpoint (`step_175`) exists.
* Eval stack already reads a continuous $p_{coop}$ from decision-token logprobs
  (`metastability_eval/harness/vllm_client.py`); branch `metastability-eval`
  holds the drift/dwell/Markov analyses.


## 2. The theory mapping: donors game → CRLD

| donorSim | CRLD (`llm_dynamics/donors_crld.py`) |
|---|---|
| payoffs $(b,c)$, both players donors | donation-game PD $(R,T,S,P)=(b-c,\;b,\;-c,\;0)$; on the training's normalized scale $\big(\tfrac{b}{b+c},1,0,\tfrac{c}{b+c}\big)$ (default) |
| $w$ — re-encounter probability | discount factor $\gamma=w$ (clamped to 0.99). Continuation probability *is* the discount factor of the repeated game; Nowak's $w>c/b$ is the discounted-game threshold |
| $q$ — reputation availability | observability blend $O = q\,O_{\text{full}} + (1-q)\,O_{\text{self}}$ — the partner's action is visible w.p. $q$, else only one's own |
| per-partner memory | `HistoryEmbedded(env, h=(m,m,m))` |
| set-strategy opponent | pinned policy $X_{opp}[s,a]$ (AllC/AllD/TFT/TF2T/Grim-$m$/WSLS, $\epsilon$-mixed); the learner's seat gets a **fixed-opponent flow field** |

Two backgrounds are therefore available for every $(b,c,w,q,m)$:
1. **two-agent CRLD flow** (both seats learning) in the *cooperation plane*
   $(P_{model}(C),\,P_{partner}(C))$ — the repo convention;
2. **fixed-opponent flow** in the *reciprocity plane*
   $\big(P(C\mid\text{sustained mutual }C),\;P(C\mid\text{own }C,\text{ partner }D)\big)$ —
   the theoretically matched background for "set strategy vs LLM".


In [ ]:
# Live: the CRLD flow of the donors game at the sweep's parameters, both backgrounds.
import os, sys, json, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd())=='llm_dynamics' else os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)
from llm_dynamics import donors_crld as dc, plots
from pyCRLD.Utils import FlowPlot as fp
b, c, w, q, m = 4.0, 2.0, 0.75, 0.75, 1
memo = dc.donors_memo_env(b, c, memory=m, q=q); mae = dc.build_mae(memo, w=w, q=q)
fig, axs = plt.subplots(1, 2, figsize=(10, 4.6))
plots.crld_flow_background(axs[0], mae, dc.allc_state(memo))
axs[0].set_title(f'two-agent CRLD flow (b={b:g}, c/b={c/b:g}, γ=w={w}, q={q}, memory {m})', fontsize=9)
axs[0].set_xlabel('agent 0 P(C)'); axs[0].set_ylabel('agent 1 P(C)')
X_opp = dc.strategy_policy('tit_for_tat', memo)
XX, YY, dX, dY = dc.fixed_opponent_flow(mae, memo, X_opp, dc.uniform_state(memo,'c','c',m), dc.uniform_state(memo,'c','d',m), NrRandom=6)
mdx, mdy = dX.mean(-1), dY.mean(-1); L = np.sqrt(mdx**2+mdy**2)+1e-12
axs[1].quiver(XX, YY, mdx*np.sqrt(L)/L, mdy*np.sqrt(L)/L, color='0.6', angles='xy')
axs[1].set_title('fixed-opponent flow vs TFT (reciprocity plane)', fontsize=9)
axs[1].set_xlabel('P(C | sustained mutual C)'); axs[1].set_ylabel('P(C | own C, partner D)')
for a in axs: a.set_xlim(-.03,1.03); a.set_ylim(-.03,1.03)
plt.tight_layout(); plt.show()

## 3. How the LLM is measured

* **Order parameter.** `p_cooperate` = renormalized COOPERATE-mass vs DEFECT-mass
  in the top-20 logprobs at the token after the *last* `DECISION:` marker
  (`llm_client.decision_probability`). It reads the distribution the sampler
  drew from; decoding is unchanged. Because the model writes a rationale before
  `DECISION:` (sampled at $T=0.6$), a single call's $p$ is near-binary — the
  stochasticity lives in the rationale — so means over seeds/rounds are needed.
* **Probes** (`policy_probe.py`): render the exact prompt for every memory-$m$
  history state ($4^m$ states) and read $p$ greedily → the LLM as a point
  $X[s]$ in CRLD policy space. Donors framing = conversational replay with forced
  `DECISION:` assistant turns; matrix framing = the stateless memory-$m$ prompt.
* **Play** (`donors_game.py`, `matrix_games.py`): full games vs a fixed
  strategy, per-round JSONL (`model_action`, `opp_action`, `p_cooperate`, HKB
  $\phi$, payoffs on both scales, swap events, visible state).
* **Portraits** (`plots.py`): cooperation-plane paths (sliding window 8) over the
  two-agent flow; reciprocity-plane stars (probes) + paths (windowed conditional
  frequencies) over the fixed-opponent flow; $q\times w$ heatmaps with the
  $q=c/b$ line.
* **Optimality gaps** (`analysis.py`), two optima by dynamic programming over
  the fixed strategy's state machine, per partner segment (swaps wipe the
  opponent's memory), same horizon:
  - **best-response captured** = model payoff ÷ the max payoff an oracle could
    extract from that strategy ("against AllD the model should AllD" ⇔ 1.0);
  - **welfare captured** = joint payoff ÷ the max joint (model + partner) payoff
    reachable against that strategy — the utilitarian point of the Pareto
    frontier the model controls (mutual cooperation vs TFT/AllC). The two
    disagree exactly where reciprocity and altruism disagree: vs AllD the
    welfare optimum is to be exploited (joint 6/round vs 4), so a perfect
    reciprocator scores BR 1.00 / welfare 0.67.

### 3.0 Opponent strategies (all nine)

| strategy | rule | reference behaviour |
|---|---|---|
| AllC | C every round | all C |
| AllD | D every round | all D (all C in Chicken / Harmony, where $S>P$) |
| TFT | opens C, mirrors your last move | all C |
| TF2T | opens C, defects after two consecutive D | all C |
| Grim | C until your first D, then D forever | all C |
| Random | C/D with prob ½ | D |
| suspicious TFT | opens **D**, then mirrors | all C |
| WSLS (Pavlov) | repeat if you cooperated, switch if you defected | all C |
| generous TFT | TFT, forgives a D with prob 0.3 | all C |

The first six are donorSim's training pool (`OPPONENT_POOL`); the last three
are added here. Each is also a pinned CRLD policy (fixed-opponent flows) and a
finite-state machine for the dynamic programs.

### 3.1 Fidelity to the training environment (deliberate choices)
* Donors prompts are byte-faithful to the `direct_indirect` rollout (rules,
  $w$/$q$ phrasing, $q$-gated reputation report, `/no_think`, round-result
  turns, swap text). Strategies, payoffs, HKB, parsing are vendored verbatim
  (`strategies.py`, from `game_helpers.py` @ merge-base `e665f87`).
* **Deviation:** `--swap-mode same` — a successor partner keeps the *same*
  strategy so the dyad stays strategy-controlled (`pool` reproduces training).
* **Addition:** `suspicious_tit_for_tat` (opens D, then mirrors) — not in the
  training pool; used so that half of the TFT games open with D.
* **Addition:** an LLM memory window for the donors framing (`--memories`):
  the rules prompt + only the last $m$ (assistant, round-result) turn pairs.
  `full` is the training setting.
* Matrix games are **stateless** per round with an explicit last-$m$ history
  block, so the LLM is genuinely a memory-$m$ policy on exactly the CRLD state
  space. Payoffs: PD (3,5,0,1), Chicken (3,5,1,0), Stag Hunt (5,3,0,1),
  Harmony (5,3,2,0) as $(R,T,S,P)$.
* Qwen3 thinking mode is **off** (`enable_thinking=False` + `/no_think`), the
  stage-1 training setting. Temperature 0.6 for play, 0 for probes.

### 3.2 Compute rule
**Everything GPU runs through SLURM** (`llm_dynamics/slurm/*.sbatch`), never on
`rnn`. Server: `serve_qwen32b.sbatch` (vllm-env, one A100-80GB, writes
`logs/vllm_endpoint.txt` + `.ready`). Clients are CPU jobs that wait on the
sentinel. `llm_dynamics/README.md` has the commands.


## 4. Experiment registry — `qwen8b_base` (v3)

    Design (identical for base and trained checkpoints, so tables are directly comparable):

    * donors game (training-faithful prompts): 9 strategies × (w,q) ∈ {0, .5, 1}² × LLM memory
      {full, 1, 2, note2} × 4 seeds × 20 rounds at c/b = 0.5; c/b ∈ {0.25, 0.75, 1.0, 1.25} at
      w ∈ {0.5, 1}, q ∈ {0, 1}; horizons N ∈ {5, 10}; thinking-on subset at w = 1
    * perturbation/repair: forced defection at round 8 vs TFT, Grim, WSLS, TF2T, generous TFT
    * self-play (model vs itself) at (w,q) ∈ {0, 1}²
    * group-selection stage (the trained environment, thinking on): 50 scenarios × 2 seeds,
      PREDICT + DECISION, CFE / Brier / ρ / bonus / trajectory scalar
    * matrix games: PD, Chicken, Stag Hunt, Harmony × memory {1, 2, 3} × 9 strategies × 4 seeds
    * probes to memory 3, both framings

    Outputs: `results/qwen8b_base_*_v3*/`, `results/probes/qwen8b_base_*`, `results/qwen8b_base_v3_tables.md`.
    

## 5. Results

### 5.1 Probed conditional policy (donors framing, memory 1 and 2)

In [ ]:
def show_probes(pattern):
        rows = []
        for f in sorted(glob.glob(pattern)):
            p = json.load(open(f)); name = os.path.basename(f).replace('.json','')
            row = {'probe': name}
            row.update({k: v['p_cooperate'] for k, v in sorted(p['states'].items())})
            rows.append(row)
        return pd.DataFrame(rows).set_index('probe').round(2) if rows else 'pending'
    display(show_probes('llm_dynamics/results/probes/qwen8b_base_donors_m1_*.json'))
    display(show_probes('llm_dynamics/results/probes/qwen8b_base_donors_m2_*.json').T)
    for g in ['ipd','chicken','staghunt','harmony']:
        print(g); display(show_probes(f'llm_dynamics/results/probes/qwen8b_base_{g}_m1.json'))

### 5.2 All tables

    Per-strategy tables (cell = agreement with the reference policy · model/reference payoff ·
    best-response captured · welfare captured), unknown-partner tables (blind-policy optimum
    under equal and training-pool priors), c/b sweep, thinking-on subset, horizons, and the
    training-signal view (mean ρ · mean r₁ · std of the trajectory scalar across seeds ·
    inference latency) with the repair table. Reference = all-C vs AllC and conditional
    cooperators, all-D vs AllD (all-C in Chicken/Harmony).

In [ ]:
from IPython.display import Markdown
    f = 'llm_dynamics/results/qwen8b_base_v3_tables.md'
    display(Markdown(open(f).read()) if os.path.exists(f) else Markdown('**tables pending**'))

### 5.3 Group-selection stage (the trained environment) and self-play

In [ ]:
for f, label in [('llm_dynamics/results/qwen8b_base_group_v3/summary_*.csv', 'group stage'),
                     ('llm_dynamics/results/qwen8b_base_selfplay_v3/summary_*.csv', 'self-play')]:
        fs = glob.glob(f)
        if fs:
            df = pd.read_csv(fs[0]); print(label, f'({len(df)} games)')
            num = df.select_dtypes('number')
            display(num.mean().round(3).to_frame('mean').T)
            if label == 'self-play':
                display(df.groupby(['w','q'])[['a_cooperation_rate','b_cooperation_rate','mutual_c_rate','a_total','b_total']].mean().round(2))
            else:
                display(df.groupby('partner')[['cooperation_rate','mean_r1','mean_rho','mean_cfe','mean_brier','trajectory_scalar']].mean().round(3))

### 5.4 Arrow fields

    Companion notebooks `llm_dynamics_flow_fields_donors_qwen8b_base.ipynb` /
    `llm_dynamics_flow_fields_matrix_qwen8b_base.ipynb`; PNGs in `llm_dynamics/results/flow_grids_qwen8b_base/`.
    One representative field — the four games at memory 2:

In [ ]:
f = 'llm_dynamics/results/flow_grids_qwen8b_base/theory_matrix_m2.png'
    if os.path.exists(f): _d(Image(f, width=900))

## 6. How to compare against a trained checkpoint

    Serve it with `llm_dynamics/slurm/serve_model.sbatch` (`MODEL_PATH=<merged dir>` or
    `LORA=<adapter dir>`), run `TAG=<tag> MODEL=<served name> ENDPOINT_FILE=... sbatch
    llm_dynamics/slurm/run_suite_v3.sbatch`, then `TAG=<tag> VERSION=v3 python
    llm_dynamics/build_notebook.py --execute` and `reciprocity-figure --probes <base> <trained>`.
    